In [ ]:
using Plots
using FFTW
using LinearAlgebra

# Boundary Value Problems and the FFT

## Example 1
Solve
$$
-u'' = -\sin(x) \sin(\sin(x)) + \cos(x)^2 \cos(\sin(x))
$$
with periodic conditions by FFT methods.  The exact solution is $\cos(\sin(x))$

In [ ]:
N = 64;
x = LinRange(0, 2*pi, N+1)[1:end-1];
k = [0:N÷2-1; -N÷2:-1]; # wave numbers
ik = im * k;

# for comparison
u_exact = @. cos(sin(x));

f = @. -sin(x) * sin(sin(x)) + cos(x)^2 *cos(sin(x));

fhat = fft(f);
uhat = zeros(ComplexF64, N);
uhat[1] = 0; # set mode zero

@. uhat[2:end] = fhat[2:end] / k[2:end]^2;

u = ifft(uhat);



In [ ]:
u

In [ ]:
plot(x, real.(u), label="numerical", lw=2)
plot!(x, u_exact, label="exact", lw=2, ls=:dash)
xlabel!("x")

There is a vertical offset because the solution is not unique with respect to $\hat{u}_0$, the mean.  To do a comparison, we need to do a ``registration'':

In [ ]:
plot(x, real.(u .-u[1]), label="numerical", lw=2)
plot!(x, u_exact.-u_exact[1], label="exact", lw=2, ls=:dash)
xlabel!("x")

In [ ]:
@show norm(u .-u[1] .- (u_exact.-u_exact[1]), Inf);

## Example 2

Solve
$$
u - u'' = f
$$
with the related $f$ as above.

In [ ]:
N = 16;
x = LinRange(0, 2*pi, N+1)[1:end-1];
k = [0:N÷2-1; -N÷2:-1]; # wave numbers
ik = im * k;

# for comparison
u_exact = @. cos(sin(x));

f = @. -sin(x) * sin(sin(x)) + cos(x)^2 *cos(sin(x)) + cos(sin(x));

fhat = fft(f);
uhat = zeros(ComplexF64, N);

@. uhat = fhat / (1+ k^2);

u = ifft(uhat);
@show norm(u .-u[1] .- (u_exact.-u_exact[1]), Inf);


# Heat Equation with the FFT

## Analytic Solution

In [ ]:
N = 256;
x = LinRange(0, 2*pi, N+1)[1:end-1];
k = [0:N÷2-1; -N÷2:-1]; # wave numbers

# u0 = @. x * (2*pi - x);
u0 = @. sin(3*x).^2;
u0hat = fft(u0);

# when do we evaluate it in time?
t_vals = [0., 0.01, 0.1, 0.5, 1.0, 10.0];
u_vals = [real(ifft(u0hat .* exp.(-k.^2 * t))) for t in t_vals];


In [ ]:
plot(x, u_vals[1], label="t = $(t_vals[1])", lw=2)
for i in 2:length(t_vals)
    plot!(x, u_vals[i], label="t = $(t_vals[i])", lw=2)
end
xlabel!("x")

## DifferentialEquations Solution

In [ ]:
using DifferentialEquations

In [ ]:
function heatfft!(du, u, p, t)
    k2 = p[1];
    du .= real(ifft(-k2.*fft(u)));
    du
end

In [ ]:
tspan = (0., 10.);
u0 = @. sin(3*x).^2;

p = (k.^2,);
prob = ODEProblem(heatfft!, u0, tspan, p);
sol = solve(prob, Tsit5());

In [ ]:
tt = LinRange(0., 10., 1000);
anim = @animate for i in 1:length(tt)
    plot(x, sol(tt[i]), label="t = $(round(tt[i], digits=2))", lw=2)
    ylims!(-0.1, 1.0)
end

gif(anim, "heat_equation.gif", fps=1)


## Sovle a semilinear equation
$$
u_t = u(1-u) + u_{xx}
$$
with PBC

In [ ]:
N = 256;
x = LinRange(0, 2*pi, N+1)[1:end-1];
k = [0:N÷2-1; -N÷2:-1]; # wave numbers

# u0 = @. x * (2*pi - x);
u0 = @. 1.5 * sin(3*x).^2;

function semiheatfft!(du, u, p, t)
    k2 = p[1];
    du .= real(ifft(-k2.*fft(u))) + u.^2 .* (1 .-u);
    du
end


p = (k.^2,);
prob = ODEProblem(semiheatfft!, u0, tspan, p);
sol = solve(prob, Tsit5());

In [ ]:
tt = LinRange(0., 10., 100);
anim = @animate for i in 1:length(tt)
    plot(x, sol(tt[i]), label="t = $(round(tt[i], digits=2))", lw=2)
    ylims!(-2.0, 2.0)
end

gif(anim, fps=1)
